# Lección 8.4 — Reducción de dimensionalidad: PCA, la mejor "foto" de los datos

> **Módulo 8 · Aprendizaje no supervisado** · Nivel 3 — Practicante
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 2.8 (maldición de la dimensionalidad), Lección 2.3 (escalado), Lección 7.5 (pipelines)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía qué hace PCA: buscar el ángulo desde el que los datos se ven **más extendidos**.
- Leer la **varianza explicada** y decidir cuántos componentes conservar.
- Interpretar un componente a partir de sus **cargas** (qué columnas lo forman).
- Usar PCA para **visualizar** y como **preprocesamiento** dentro de un pipeline, y conocer sus trampas.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, load_wine, load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: la mejor foto de un objeto 📸

Tienes que mandarle a un amigo **una sola foto** de una tetera para que la reconozca. Una foto es **plana** (2D) y la tetera es **3D**: algo se pierde siempre. Pero no todas las fotos pierden lo mismo:
- Desde **arriba** ves un círculo: podría ser un plato, una olla, cualquier cosa.
- Desde un **lado** ves el pico, el asa y la tapa: ¡es claramente una tetera!

La mejor foto es la que muestra los datos **lo más "extendidos" posible**, donde se notan más las diferencias.

**PCA** (*Principal Component Analysis*, análisis de componentes principales) hace exactamente eso con datos de muchas columnas: busca el **ángulo** desde el que los datos se ven más extendidos y los "fotografía" desde ahí, usando **menos columnas** y perdiendo la menor información posible.

### ✋ Ejercicio 1 — Calentamiento
Quieres mostrar en una foto **lo largo** que es un lápiz. ¿Desde dónde lo fotografías: apuntando a la punta, o de costado? ¿Qué pasa con la otra foto?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**De costado.** Apuntando a la punta, el lápiz se ve como un puntito: toda su longitud (su dirección de mayor "extensión") desaparece en la foto. De costado conservas la información importante.

PCA elige siempre "de costado": la dirección en la que los datos **varían más**. A esa variación se le llama **varianza**.
</details>

---
## 2. 📖 El concepto: componentes principales y varianza explicada

Empecemos con solo 2 columnas: **estatura** y **peso** de 300 personas. Están muy relacionadas: quien es más alto suele pesar más.

In [ ]:
rng = np.random.default_rng(0)
estatura = rng.normal(170, 10, 300)
peso = 0.9 * (estatura - 170) + 70 + rng.normal(0, 4, 300)
personas = pd.DataFrame({"estatura": estatura, "peso": peso})

X_p = StandardScaler().fit_transform(personas)      # escalamos (verás por qué en la sección 4)
plt.scatter(*X_p.T, s=10, alpha=.6)
plt.xlabel("estatura (escalada)"); plt.ylabel("peso (escalado)"); plt.axis("equal"); plt.show()

La nube tiene forma de **cigarro inclinado**. PCA busca:
- **Componente principal 1 (PC1):** la dirección a lo largo del cigarro, donde los datos están más extendidos.
- **Componente principal 2 (PC2):** la dirección **perpendicular** a la primera (a lo ancho del cigarro).

Cada componente "explica" una parte de la varianza total. Juntos explican el 100 %.

### 🔮 Ejercicio 2
¿Qué porcentaje de la varianza crees que explica PC1? ¿Más del 50 %, del 75 %, del 90 %?

✍️ *Mi apuesta:*

In [ ]:
pca_p = PCA(n_components=2).fit(X_p)
print("Varianza explicada por cada componente:", pca_p.explained_variance_ratio_.round(3))

plt.scatter(*X_p.T, s=10, alpha=.4)
for vector, largo, color in zip(pca_p.components_, pca_p.explained_variance_, ["red", "green"]):
    plt.arrow(0, 0, *(vector * 2 * np.sqrt(largo)), width=0.05, color=color)
plt.text(1.6, 1.9, "PC1", color="red", fontsize=14); plt.text(0.7, -0.9, "PC2", color="green", fontsize=14)
plt.axis("equal"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

PC1 explica ≈ **96 %** de la varianza y PC2 solo ≈ **4 %**. Casi toda la información de "estatura y peso" cabe en **un solo número**: la posición a lo largo de la flecha roja.

¿Y qué significa ese número? La flecha roja apunta hacia "más alto **y** más pesado", así que PC1 es algo como **"tamaño corporal"**. PC2 (verde) es lo que queda: ser más pesado o más ligero **de lo esperado** para tu estatura.
</details>

### 📖 Las cargas: ¿de qué está hecho cada componente?
Cada componente es una **receta**: una mezcla de las columnas originales. Los pesos de la receta se llaman **cargas** (*loadings*) y están en `components_`.

In [ ]:
pd.DataFrame(pca_p.components_, columns=personas.columns, index=["PC1", "PC2"]).round(2)

### ✋ Ejercicio 3
Lee la tabla:
1. PC1 tiene cargas ≈ +0.71 en estatura y peso. ¿Por qué confirma que es "tamaño corporal"?
2. En PC2 una carga es positiva y la otra negativa. ¿Qué tipo de persona tendrá un PC2 muy alto?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Ambas cargas son **iguales y positivas**: PC1 sube cuando suben estatura **y** peso a la vez. Es un "tamaño general". (Los signos pueden salir todos invertidos, −0.71 y −0.71: significa lo mismo, solo que la flecha apunta al revés.)
2. Aquí PC2 tiene +0.71 en estatura y −0.71 en peso: un PC2 alto corresponde a alguien **más alto y más ligero de lo que "tocaría"** (complexión delgada); un PC2 muy negativo, a alguien bajo y pesado para su estatura. PC2 **compara** una columna contra la otra.

Interpretar componentes leyendo sus cargas es una habilidad clave: así le das **nombre** a algo que el algoritmo solo conoce como "PC1".
</details>

---
## 3. 📖 Cómo funciona: proyectar es tomar la foto

"Reducir a 1 dimensión" significa quedarse **solo** con PC1: cada persona queda resumida en **un número**. Si con ese número intentamos reconstruir las dos columnas originales, todos los puntos caen **sobre la flecha roja**: perdimos el "ancho del cigarro".

### 🔮 Ejercicio 4
Si reconstruimos los datos usando solo PC1, ¿cómo se verá la nube de puntos reconstruida?

✍️ *Mi apuesta:*

In [ ]:
pca_1 = PCA(n_components=1).fit(X_p)
resumen = pca_1.transform(X_p)                # 300 × 1: un número por persona
reconstruido = pca_1.inverse_transform(resumen)

print("Forma original:", X_p.shape, "→ forma reducida:", resumen.shape)
plt.scatter(*X_p.T, s=10, alpha=.3, label="original")
plt.scatter(*reconstruido.T, s=10, color="red", label="reconstruido con 1 componente")
plt.axis("equal"); plt.legend(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Todos los puntos reconstruidos quedan **en una línea recta** (la dirección de PC1). Es la "sombra" de cada punto sobre esa línea. Perdimos el 4 % de la varianza (el grosor del cigarro), pero pasamos de 2 números a 1 por persona.

Con 2 columnas no parece gran cosa. Con **64** o con **1,000** columnas, es enorme.
</details>

---
## 4. 📖 PCA con datos reales: imágenes de dígitos

Cada imagen de este conjunto es un dígito escrito a mano de **8 × 8 píxeles = 64 columnas**. No podemos dibujar 64 dimensiones… pero sí 2.

### 🔮 Ejercicio 5
¿Cuántos componentes (de 64) crees que hacen falta para conservar el **90 %** de la varianza?

✍️ *Mi apuesta:*

In [ ]:
digitos = load_digits()
X_d, y_d = digitos.data, digitos.target
print("Forma de los datos:", X_d.shape)

pca_d = PCA().fit(X_d)                    # todos los componentes
acumulada = np.cumsum(pca_d.explained_variance_ratio_)

plt.plot(range(1, 65), acumulada, "o-", ms=3)
plt.axhline(0.9, color="red", ls="--")
plt.xlabel("Número de componentes"); plt.ylabel("Varianza explicada acumulada"); plt.grid(alpha=.3); plt.show()
print("Componentes para el 90 %:", np.argmax(acumulada >= 0.90) + 1)
print("Componentes para el 95 %:", np.argmax(acumulada >= 0.95) + 1)

<details><summary><b>👀 Ver respuesta</b></summary>

Bastan **21 componentes** para el 90 % y **29** para el 95 %. Es decir, con un tercio de las columnas conservamos casi toda la información. Muchos píxeles (los de las orillas, casi siempre blancos) no aportan nada, y otros van siempre juntos.

> 📝 Aquí **no escalamos** porque todos los píxeles ya están en la misma unidad (intensidad de 0 a 16). Escalar inflaría la importancia de los píxeles de la orilla, que casi nunca cambian.
</details>

### ▶️ La foto en 2D
Ahora dibujamos los 1,797 dígitos usando solo PC1 y PC2, coloreados por el dígito real (que PCA **no** usó).

In [ ]:
X_2d = PCA(n_components=2).fit_transform(X_d)
plt.figure(figsize=(8, 6))
plt.scatter(*X_2d.T, c=y_d, cmap="tab10", s=8)
plt.colorbar(label="dígito real")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("64 dimensiones → 2 con PCA"); plt.show()
print(f"Varianza explicada por PC1 + PC2: {PCA(n_components=2).fit(X_d).explained_variance_ratio_.sum():.0%}")

### ✋ Ejercicio 6
1. ¿Qué dígitos se ven bien separados? ¿Cuáles se amontonan en el centro?
2. Solo conservamos el 29 % de la varianza. ¿Es sorprendente que se mezclen?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. El **0** (abajo), el **4** y el **6** (a la derecha) y el **3** (a la izquierda) forman zonas bastante reconocibles, pero otros como el **1, 7, 5 y 8** se amontonan y se mezclan en el centro.
2. No: con 2 de 64 componentes perdimos el **71 %** de la varianza. Es como la foto de la tetera desde arriba: algo se ve, pero no todo. En la Lección 8.5 veremos **t-SNE**, que hace una "foto" 2D mucho más clara de estos mismos dígitos.
</details>

### ✏️ Ejercicio 7 — ¿Cuántos componentes necesita un "3"?
Reconstruimos un dígito usando solo los primeros `N` componentes. Cambia `N_COMPONENTES` y observa a partir de cuántos se reconoce el número.

In [ ]:
N_COMPONENTES = [2, 5, 10, 20, 64]     # ✏️ prueba otros valores
ejemplo = X_d[3:4]                      # la imagen número 3 del conjunto

fig, axes = plt.subplots(1, len(N_COMPONENTES), figsize=(3 * len(N_COMPONENTES), 3))
for ax, n in zip(axes, N_COMPONENTES):
    p = PCA(n_components=n).fit(X_d)
    ax.imshow(p.inverse_transform(p.transform(ejemplo)).reshape(8, 8), cmap="gray_r")
    ax.set_title(f"{n} componentes\n({p.explained_variance_ratio_.sum():.0%} varianza)"); ax.axis("off")
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Con 2 y 5 componentes es una mancha borrosa; con 10 ya se intuye un **3**, y con 20 (≈89 % de la varianza) se parece bastante al original (64). Es **compresión**: guardar 20 números en vez de 64 y reconstruir una imagen muy parecida.
</details>

---
## 5. 📖 ¿Qué pasa si… no escalas?

PCA busca la dirección de **mayor varianza**. Si una columna se mide en números enormes, su varianza será enorme… y PCA creerá que es lo único importante.

### 🔮 Ejercicio 8
Los vinos de la Lección 8.1 tienen 13 columnas, y `proline` llega a valores de 1,600 mientras las demás están por debajo de 30 (salvo `magnesium`, ~160). Sin escalar, ¿cuánta varianza explicará PC1? ¿Qué columna dominará su receta?

✍️ *Mi apuesta:*

In [ ]:
vinos = load_wine(as_frame=True)
X_v = vinos.data

for nombre, datos in [("Sin escalar", X_v), ("Escalado", StandardScaler().fit_transform(X_v))]:
    p = PCA(n_components=2).fit(datos)
    cargas_pc1 = pd.Series(np.abs(p.components_[0]), index=X_v.columns).sort_values(ascending=False)
    print(f"{nombre:12} → varianza PC1 = {p.explained_variance_ratio_[0]:.1%} · "
          f"columnas que más pesan en PC1: {list(cargas_pc1.index[:3])}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **Sin escalar:** PC1 explica el **99.8 %** de la varianza y su receta es casi solo `proline`. Parece un resumen "perfecto"… pero es una ilusión: PC1 **es** `proline`, y las otras 12 columnas quedaron ignoradas.
- **Escalado:** PC1 explica un 36 % más honesto, y su receta mezcla varias columnas químicas (`flavanoids`, `total_phenols`, `od280…`): ahora sí es un resumen de **muchas** medidas.

Regla: **escala antes de PCA** salvo que todas las columnas estén en la misma unidad (como los píxeles).
</details>

---
## 6. 📖 Hiperparámetros y PCA como preprocesamiento

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `n_components` (entero) | Cuántos componentes conservar | Más información, menos compresión | Más compresión, se pierde información | 2–3 para **visualizar** |
| `n_components` (entre 0 y 1) | Conserva los necesarios para esa fracción de varianza | Más componentes | Menos componentes | 0.90–0.95 para **preprocesar** |
| `whiten` | Reescala los componentes para que todos tengan varianza 1 | — | — | `False` (útil antes de algunos modelos) |
| `svd_solver` | Método de cálculo | — | — | `"auto"`; `"randomized"` es más rápido con datos grandes |

### 📖 PCA antes de un modelo supervisado
PCA puede ir **dentro de un pipeline** (Lección 7.5), antes de un clasificador: comprime las columnas, quita ruido y acelera el entrenamiento. Como está dentro del pipeline, se aprende solo con los datos de entrenamiento de cada pliegue: **sin fuga de datos** (Lección 2.6).

### 🔮 Ejercicio 9
Clasificaremos los dígitos con una regresión logística, primero con 2 componentes, luego con 5, 10, 20, 40 y las 64 columnas. ¿Cómo crees que será la curva de exactitud? ¿Siempre sube?

✍️ *Mi apuesta:*

In [ ]:
for n in [2, 5, 10, 20, 40, 64]:     # ✏️
    modelo = make_pipeline(StandardScaler(), PCA(n_components=n), LogisticRegression(max_iter=2000))
    exactitud = cross_val_score(modelo, X_d, y_d, cv=5).mean()
    print(f"{n:>2} componentes → exactitud CV = {exactitud:.3f}  " + "█" * int(exactitud * 40))

<details><summary><b>👀 Ver respuesta</b></summary>

Con 2 componentes la exactitud es pobre (≈**0.53**), sube rápido (≈0.77 con 5, ≈0.84 con 10) y con **20 componentes** (≈0.90) ya está cerca de usar las 64 columnas (≈0.92). De 40 a 64 casi no se gana nada (0.91 → 0.92): los últimos componentes son, sobre todo, ruido.

Es la ley de los rendimientos decrecientes: un tercio de las columnas da casi toda la exactitud, con un modelo más pequeño y rápido.

> 📝 Aquí escalamos dentro del pipeline (algo que en la sección 4 evitamos para dibujar). Para el clasificador funciona bien de las dos maneras; prueba quitar `StandardScaler()` si tienes curiosidad.
</details>

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| PC1 explica el 99 % y su receta es una sola columna | Datos **sin escalar** | `StandardScaler` antes de PCA |
| Resultados demasiado buenos en prueba | PCA ajustado con **todos** los datos antes de dividir (fuga) | Ponerlo dentro de un `Pipeline` (Lección 7.5) |
| "PC1 es el ingreso" | Interpretar un componente como **una** columna | Leer sus cargas: es una **mezcla** |
| El modelo empeora mucho tras PCA | Se descartó una dirección de **poca varianza pero muy predictiva** | Probar más componentes o no usar PCA (ver ejercicio de abajo) |
| La foto 2D no muestra grupos que sí existen | La estructura es **no lineal** (curvas, espirales) | t-SNE o UMAP para visualizar (Lección 8.5) |
| Componentes imposibles de explicar a negocio | PCA mezcla todas las columnas | Usar menos columnas, o seleccionar variables en vez de comprimir |

### 🔮 Ejercicio 10 — Poca varianza ≠ poca importancia
Un sensor mide dos cosas de unas piezas: `largo` (varía mucho, ±10 mm, pero no tiene que ver con los defectos) y `grosor` (varía muy poco, ±0.3 mm, y **las piezas defectuosas son un poco más delgadas**). Reducimos a **1** componente con PCA y luego clasificamos. ¿Qué crees que pasará?

✍️ *Mi apuesta:*

In [ ]:
rng = np.random.default_rng(1)
defecto = rng.integers(0, 2, 400)
piezas = pd.DataFrame({
    "largo":  rng.normal(100, 10, 400),                        # mucha varianza, nada que ver con el defecto
    "grosor": 5 - 0.5 * defecto + rng.normal(0, 0.15, 400),    # poca varianza, ¡pero aquí está la señal!
})

con_pca = make_pipeline(PCA(n_components=1), LogisticRegression())
sin_pca = make_pipeline(StandardScaler(), LogisticRegression())
print(f"Con PCA (1 componente, sin escalar): {cross_val_score(con_pca, piezas, defecto, cv=5).mean():.1%}")
print(f"Sin PCA (2 columnas escaladas)     : {cross_val_score(sin_pca, piezas, defecto, cv=5).mean():.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

Con PCA la exactitud cae a ≈**56 %** (¡casi como lanzar una moneda!), mientras que sin PCA es ≈**94 %**. PCA se quedó con el `largo` porque es lo que **más varía**, y tiró el `grosor`, que es justo donde estaba la información sobre los defectos.

PCA **no sabe** qué quieres predecir: solo busca varianza. Por eso, cuando lo uses antes de un modelo, **compara siempre** con y sin PCA usando validación cruzada.
</details>

---
## 8. 📋 Ficha del modelo

| | PCA (análisis de componentes principales) |
|---|---|
| **Tipo** | No supervisado: reducción de dimensionalidad (lineal) |
| **Intuición** | Fotografiar los datos desde el ángulo en que se ven más extendidos |
| **Cómo funciona** | Encuentra direcciones perpendiculares de máxima varianza (los componentes) y proyecta los datos sobre las primeras |
| **Hiperparámetros clave** | `n_components` (entero o fracción de varianza) |
| **✅ Fortalezas** | Rápido y determinista · Cuantifica cuánta información conservas · Comprime y quita ruido · `transform` funciona con datos nuevos · Cargas interpretables |
| **❌ Debilidades** | Solo capta estructura **lineal** · Sensible a la escala · Componentes = mezclas difíciles de nombrar · Ignora la variable objetivo |
| **Úsalo cuando…** | Tienes muchas columnas correlacionadas, quieres una primera foto 2D, o comprimir antes de un modelo |
| **Evítalo cuando…** | La estructura es curva (usa t-SNE/UMAP para ver) o necesitas explicar cada columna original |

## 9. 🧠 Resumen en 5 puntos
1. PCA busca nuevas direcciones (**componentes**) donde los datos **varían más**, ordenadas de mayor a menor.
2. La **varianza explicada** dice cuánta información conserva cada componente; elige cuántos con la curva acumulada (90–95 %).
3. Cada componente es una **receta** de columnas originales (cargas): léelas para darle nombre.
4. **Escala** antes de PCA (salvo columnas en la misma unidad) y úsalo **dentro de un pipeline** para evitar fugas.
5. PCA es **lineal** y **no mira el objetivo**: poca varianza no significa poca importancia.

---
## 10. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 10.1 Repaso rápido (opción múltiple)

**1.** PC1 explica el 70 % de la varianza y PC2 el 20 %. ¿Cuánto conservas si te quedas con los dos?
- a) 70 % · b) 90 % · c) 20 % · d) No se puede saber

**2.** ¿Qué tienen siempre los componentes principales entre sí?
- a) Son paralelos · b) Son perpendiculares (independientes) · c) Tienen la misma varianza · d) Son columnas originales

**3.** Aplicas PCA a datos con `ingreso` (pesos) y `edad` (años) sin escalar. ¿Qué esperas?
- a) PC1 ≈ ingreso · b) PC1 ≈ edad · c) PC1 = mitad y mitad · d) Error

**4.** `PCA(n_components=0.95)` significa…
- a) Usar 95 componentes · b) Conservar los componentes necesarios para el 95 % de la varianza · c) Quitar el 95 % de las filas · d) Usar el 95 % de las columnas

**5.** ¿Cuál de estos **no** es un buen uso de PCA?
- a) Dibujar en 2D datos de 50 columnas · b) Comprimir imágenes · c) Elegir qué columna original es más importante para predecir ventas · d) Quitar ruido antes de un modelo

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Las varianzas explicadas se suman: 70 + 20 = 90 %.
2. **b)** Cada componente es perpendicular a los anteriores: captura información **nueva**, no repetida.
3. **a)** El ingreso tiene una varianza millones de veces mayor que la edad.
4. **b)** Un número entre 0 y 1 se interpreta como fracción de varianza.
5. **c)** PCA no mira el objetivo (ventas) y mezcla columnas. Para importancia de variables usa las herramientas de la Lección 7.6.
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Un compañero dice: "Apliqué PCA a mis 200 columnas, me quedé con 10 componentes que explican el 95 % y mi modelo empeoró un poco. PCA no sirve." ¿Qué le preguntarías y qué le sugerirías?

**B. Caso de análisis 💼** Una cadena de tiendas tiene 40 indicadores por sucursal (ventas por categoría, visitas, ticket promedio, devoluciones…). El director quiere "un mapa de las sucursales" para ver cuáles se parecen. ¿Cómo usarías PCA? ¿Qué le mostrarías y cómo le explicarías los ejes?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Preguntas: ¿escalaste antes?, ¿ajustaste PCA dentro del pipeline?, ¿el 5 % descartado podría contener la señal (como el grosor de las piezas)? Sugerencias: comparar con validación cruzada varios `n_components`; si con todas las columnas el modelo es mejor y el tiempo no es problema, **no uses PCA** para predecir. PCA no es obligatorio: es una herramienta para comprimir, visualizar o quitar ruido, y "empeorar un poco" puede ser un precio aceptable si el modelo es mucho más rápido.

**B.** Escalar los 40 indicadores, aplicar PCA con 2 componentes y dibujar cada sucursal como un punto (con su nombre). Reportar cuánta varianza explican los dos ejes (por ejemplo: "este mapa resume el 60 % de las diferencias") y **nombrar cada eje leyendo sus cargas**: p. ej. PC1 = "tamaño de la tienda" (todas las ventas suben juntas), PC2 = "perfil electrónica vs. abarrotes". Sucursales cercanas en el mapa se parecen. Se puede combinar con K-means (Lección 8.2) para colorear grupos de sucursales.
</details>

### 10.3 🧩 Práctica integradora: "Resumir 30 medidas de un tumor"

Retomamos el diagnóstico de cáncer de mama de la Lección 5.4: 569 pacientes, **30 medidas** de las células (radio, textura, perímetro, área… cada una con su promedio, su error y su peor valor). Los médicos preguntan: *"¿Hacen falta las 30? ¿Podemos verlas en un solo gráfico?"*

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Escala y mira la varianza explicada**

In [ ]:
cancer = load_breast_cancer(as_frame=True)
X_c, y_c = cancer.data, cancer.target           # y: 0 = maligno, 1 = benigno (PCA no lo usa)
X_c_esc = StandardScaler().fit_transform(X_c)

pca_c = PCA().fit(X_c_esc)
acum_c = np.cumsum(pca_c.explained_variance_ratio_)
for n in [1, 2, 3, 5, 7, 10]:
    print(f"{n:>2} componentes → {acum_c[n - 1]:.0%} de la varianza")

✋ ¿Cuántos componentes necesitas para el 90 %? ¿Qué te dice eso sobre las 30 medidas?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Con **7 componentes** ya superas el 90 % (≈91 %), y con solo 2 tienes ≈63 %. Muchas de las 30 medidas son **redundantes**: el radio, el perímetro y el área, por ejemplo, dicen casi lo mismo (un tumor de radio grande tiene perímetro y área grandes).
</details>

**Paso 2 🔮 — El mapa 2D.** Dibujamos a las 569 pacientes con PC1 y PC2, coloreadas por el diagnóstico real. ¿Crees que se separarán bien los tumores malignos de los benignos?

✍️ *Mi apuesta:*

In [ ]:
X_c_2d = PCA(n_components=2).fit_transform(X_c_esc)
for clase, color, nombre in [(0, "red", "maligno"), (1, "green", "benigno")]:
    plt.scatter(*X_c_2d[y_c == clase].T, s=10, alpha=.6, c=color, label=nombre)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend(); plt.title("30 medidas → 2 componentes"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

**Sí, bastante bien**: los benignos se concentran a un lado de PC1 y los malignos se extienden hacia el otro, con una zona de mezcla en la frontera. PCA **no conocía** el diagnóstico; simplemente la dirección de mayor variación coincide con lo que distingue a los tumores.
</details>

**Paso 3 ✋ — Ponle nombre a PC1.** Mira las 6 cargas más grandes (en valor absoluto) de PC1.

In [ ]:
cargas = pd.Series(pca_c.components_[0], index=X_c.columns)
cargas.reindex(cargas.abs().sort_values(ascending=False).index).head(6).round(3)

✍️ *¿Qué nombre le darías a PC1?*

<details><summary><b>👀 Ver respuesta</b></summary>

Las cargas más grandes son de medidas de **concavidad** (`mean concave points`, `mean concavity`, `worst concave points`), **compacidad** y **perímetro/radio**, todas con el **mismo signo** y de tamaño parecido (≈0.22–0.26). PC1 es algo como **"qué tan grande e irregular es el tumor"**. Por eso separa tan bien: los tumores malignos suelen ser más grandes y de contorno más irregular. (Recuerda que en la Lección 5.4 la primera pregunta del árbol fue sobre `mean concave points`.)
</details>

**Paso 4 🔮 — ¿Cuánto se pierde al clasificar?** Comparamos una regresión logística con las 30 columnas contra la misma con solo 2 o 7 componentes (todo en un pipeline, con validación cruzada).

✍️ *Mi apuesta (¿cuántos puntos se pierden con 2 componentes?):*

In [ ]:
for n in [2, 7, None]:      # None = sin PCA
    pasos = [StandardScaler()] + ([PCA(n_components=n)] if n else []) + [LogisticRegression(max_iter=2000)]
    exactitud = cross_val_score(make_pipeline(*pasos), X_c, y_c, cv=5).mean()
    print(f"{'Sin PCA (30 columnas)' if n is None else f'PCA con {n} componentes':24} → exactitud CV = {exactitud:.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- 2 componentes: ≈**0.95**
- 7 componentes: ≈**0.97**
- 30 columnas: ≈**0.98**

¡Con solo **2** números por paciente se pierden apenas unos 3 puntos! Con 7 la diferencia es de 1 punto. Pero ojo: en un problema médico, 1–3 puntos pueden ser varias pacientes mal diagnosticadas. Que convenga comprimir depende de para qué (visualizar: sí; diagnosticar: quizá no).
</details>

**Paso 5 ✋ — Responde a los médicos.** En dos o tres frases: ¿hacen falta las 30 medidas? ¿Para qué les sirve el gráfico 2D?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

*"Las 30 medidas son muy redundantes: 7 combinaciones resumen el 91 % de la información, y con solo 2 ya se distinguen bastante bien los tumores malignos de los benignos. El gráfico 2D sirve para ver de un vistazo dónde cae un caso nuevo respecto a los anteriores (en la zona benigna, la maligna o la frontera dudosa). Para el diagnóstico automático, sin embargo, conviene usar todas las medidas: la compresión cuesta 1–3 puntos de exactitud."*
</details>

---
## 📚 Para profundizar
- 🎮 **Interactivo:** *Principal Component Analysis Explained Visually* (Setosa.io): gira los datos en 3D y mira cómo cambian los componentes.
- 📺 **Video:** StatQuest, "Principal Component Analysis (PCA), Step-by-Step" (inglés, con subtítulos).
- 📘 **Documentación:** scikit-learn, guía de usuario, sección *Decomposing signals in components* (PCA, `IncrementalPCA` para datos que no caben en memoria, `KernelPCA` para estructura no lineal).

---
⬅️ **Anterior:** [8.3 Clustering jerárquico y DBSCAN: árboles genealógicos y multitudes](8.3-jerarquico-y-dbscan.ipynb) · ➡️ **Siguiente:** [8.5 Visualización: t-SNE y UMAP, el plano de mesas de una boda](8.5-tsne-y-umap.ipynb)